In [2]:
"""
Test whether the outsourcing wage penalty changed over time, and whether
minimum-wage exposure is associated with that change.

Run:
    python test_penalty_hypotheses.py

Inputs:
    outputs/py/dk_clean.parquet
    data/minimumwage_annual.csv

Outputs:
    outputs/py/hypothesis_trend_tests.csv
    outputs/py/hypothesis_minwage_tests.csv

Penalty comparisons use five-year bins from 2001 onward. 2020 is excluded,
so 2016-2019 is a four-year bin. The minimum wage is merged annually by
state and year; it is not averaged into five-year bins.

A positive trend coefficient means the outsourcing coefficient moves upward
toward zero over time (the penalty narrows). A positive minimum-wage
interaction means the outsourcing coefficient is less negative where the
binding minimum wage is higher. These are descriptive associations.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import pyfixest as pf


# Update this if your project folder is stored elsewhere.
PROJECT = Path("/Users/kq/Projects/outsourcing-penalty")
DATA = PROJECT / "outputs/py/dk_clean.parquet"
MINWAGE = PROJECT / "data/minimumwage_annual.csv"
OUT = PROJECT / "outputs/py"
OUT.mkdir(parents=True, exist_ok=True)


COVARIATES = [
    "union",
    "parttime",
    "educ_cat",
    "female",
    "black",
    "hispanic",
    "age",
    "age2",
    "msa",
    "central_city",
    "cc_unknown",
    "geo_unidentified",
]

# These bins exclude 2020. The 2016-2019 period is four years.
EXTENSION_BINS = [
    (2001, 2005),
    (2006, 2010),
    (2011, 2015),
    (2016, 2019),
    (2021, 2025),
]
REFERENCE_PERIOD = "2001-2005"

BASE_FORMULA = (
    "log_wage ~ outsourced + "
    + " + ".join(COVARIATES)
    + " | STATEFIP^YEAR"
)


def fit_model(formula, data, cluster="ym"):
    """Fit the weighted model with standard errors clustered by survey month."""
    return pf.feols(
        formula,
        data=data,
        weights="EARNWT",
        vcov={"CRV1": cluster},
    )


def coefficient_row(fit, term):
    """Return a coefficient estimate, standard error, and p-value."""
    row = fit.tidy().loc[term]
    return {
        "estimate": float(row["Estimate"]),
        "se": float(row["Std. Error"]),
        "t": float(row["t value"]),
        "p": float(row["Pr(>|t|)"]),
        "n": int(fit._N),
    }


def joint_wald_zero(fit, terms):
    """Joint Wald test that all listed coefficients equal zero."""
    coefficient_names = list(fit.coef().index)
    missing = set(terms) - set(coefficient_names)
    if missing:
        raise ValueError(
            f"Terms missing from fitted model: {sorted(missing)}"
        )

    restriction = np.zeros((len(terms), len(coefficient_names)))
    for row_number, term in enumerate(terms):
        restriction[row_number, coefficient_names.index(term)] = 1

    result = fit.wald_test(R=restriction, distribution="chi2")
    return {
        "statistic": float(result["statistic"]),
        "df": len(terms),
        "p": float(result["pvalue"]),
    }


# ---------------------------------------------------------------------
# Load and prepare the cleaned CPS analysis sample.
# ---------------------------------------------------------------------

if not DATA.exists():
    raise FileNotFoundError(f"Cleaned CPS data not found: {DATA}")
if not MINWAGE.exists():
    raise FileNotFoundError(f"Minimum-wage CSV not found: {MINWAGE}")

needed_columns = list(dict.fromkeys(
    [
        "occupation",
        "YEAR",
        "STATEFIP",
        "outsourced",
        "EARNWT",
        "ym",
        "log_wage",
        "extension_sample",
    ]
    + COVARIATES
))

df = pd.read_parquet(DATA, columns=needed_columns)

# Limit the analysis to the 2001-2025 extension sample.
df = df[df["extension_sample"]].copy()

model_variables = list(dict.fromkeys(
    [
        "occupation",
        "YEAR",
        "STATEFIP",
        "outsourced",
        "EARNWT",
        "ym",
        "log_wage",
    ]
    + COVARIATES
))
df = df.dropna(subset=model_variables).copy()

# Standardize types for the state-year minimum-wage merge.
df["YEAR"] = df["YEAR"].astype(int)
df["STATEFIP"] = df["STATEFIP"].astype(int)


# ---------------------------------------------------------------------
# Test 1: Do penalties differ across five-year periods? Is there a trend?
# ---------------------------------------------------------------------

period_by_year = {
    year: f"{start}-{end}"
    for start, end in EXTENSION_BINS
    for year in range(start, end + 1)
}

df["period"] = df["YEAR"].map(period_by_year)
df = df.dropna(subset=["period"]).copy()

# Create outsourced-by-period interactions. The 2001-2005 bin is the
# reference period; each interaction estimates a difference from that bin.
period_interactions = {}
for start, end in EXTENSION_BINS:
    label = f"{start}-{end}"
    if label == REFERENCE_PERIOD:
        continue

    term = f"out_x_{start}_{end}"
    period_interactions[label] = term
    df[term] = df["outsourced"] * (df["period"] == label)

# Center year at 2001. Its coefficient is the annual trend in the
# outsourcing differential; below it is reported per decade.
df["out_x_year"] = df["outsourced"] * (df["YEAR"] - 2001)

trend_rows = []

for occupation in ["janitor", "guard"]:
    sample = df[df["occupation"] == occupation].copy()

    # Joint test: are all later-period penalties equal to the 2001-2005
    # penalty? This tests period differences as a group.
    interaction_terms = list(period_interactions.values())
    formula_period = (
        "log_wage ~ outsourced + "
        + " + ".join(interaction_terms)
        + " + "
        + " + ".join(COVARIATES)
        + " | STATEFIP^YEAR"
    )

    fit_period = fit_model(formula_period, sample)
    joint_result = joint_wald_zero(fit_period, interaction_terms)

    trend_rows.append({
        "occupation": occupation,
        "test": "Period differences from 2001-2005 are jointly zero",
        "estimate": np.nan,
        "se": np.nan,
        "t": np.nan,
        "statistic": joint_result["statistic"],
        "df": joint_result["df"],
        "p": joint_result["p"],
        "n": int(fit_period._N),
    })

    # Linear trend: a positive coefficient means the penalty narrows.
    formula_trend = BASE_FORMULA.replace(
        "outsourced + ",
        "outsourced + out_x_year + ",
        1,
    )
    fit_trend = fit_model(formula_trend, sample)
    trend = coefficient_row(fit_trend, "out_x_year")

    trend_rows.append({
        "occupation": occupation,
        "test": "Linear trend in penalty, per decade",
        "estimate": 10 * trend["estimate"],
        "se": 10 * trend["se"],
        "t": trend["t"],
        "statistic": np.nan,
        "df": np.nan,
        "p": trend["p"],
        "n": trend["n"],
    })

trend_results = pd.DataFrame(trend_rows)
trend_path = OUT / "hypothesis_trend_tests.csv"
trend_results.to_csv(trend_path, index=False)

print("\nTrend tests")
print(trend_results.round(4).to_string(index=False))
print(f"\nSaved: {trend_path}")


# ---------------------------------------------------------------------
# Prepare the binding minimum wage by state and year.
# ---------------------------------------------------------------------

# State postal abbreviations mapped to CPS STATEFIP codes.
FIPS = {
    "AL": 1, "AK": 2, "AZ": 4, "AR": 5, "CA": 6, "CO": 8, "CT": 9,
    "DE": 10, "DC": 11, "FL": 12, "GA": 13, "HI": 15, "ID": 16,
    "IL": 17, "IN": 18, "IA": 19, "KS": 20, "KY": 21, "LA": 22,
    "ME": 23, "MD": 24, "MA": 25, "MI": 26, "MN": 27, "MS": 28,
    "MO": 29, "MT": 30, "NE": 31, "NV": 32, "NH": 33, "NJ": 34,
    "NM": 35, "NY": 36, "NC": 37, "ND": 38, "OH": 39, "OK": 40,
    "OR": 41, "PA": 42, "RI": 44, "SC": 45, "SD": 46, "TN": 47,
    "TX": 48, "UT": 49, "VT": 50, "VA": 51, "WA": 53, "WV": 54,
    "WI": 55, "WY": 56,
}

mw = pd.read_csv(MINWAGE)
mw["YEAR"] = mw["observation_date"].astype(str).str[:4].astype(int)

# Exclude the federal series and non-state/territory series from state columns.
excluded_columns = {
    "STTMINWGFG",  # federal minimum
    "STTMINWGGU",  # Guam
    "STTMINWGPR",  # Puerto Rico
}
state_columns = [
    column
    for column in mw.columns
    if column.startswith("STTMINWG") and column not in excluded_columns
]

# Convert available state minimum-wage series to long format.
state_minimum = mw.melt(
    id_vars="YEAR",
    value_vars=state_columns,
    var_name="series",
    value_name="state_minimum",
)
state_minimum["postal"] = state_minimum["series"].str.replace(
    "STTMINWG",
    "",
    regex=False,
)
state_minimum["STATEFIP"] = state_minimum["postal"].map(FIPS)

if state_minimum["STATEFIP"].isna().any():
    unmapped = sorted(
        state_minimum.loc[
            state_minimum["STATEFIP"].isna(), "postal"
        ].unique()
    )
    raise ValueError(f"Unmapped state abbreviations: {unmapped}")

state_minimum["STATEFIP"] = state_minimum["STATEFIP"].astype(int)

federal_minimum = mw[["YEAR", "STTMINWGFG"]].rename(
    columns={"STTMINWGFG": "federal_minimum"}
)

# Build a complete grid because not every state has its own series in the CSV.
# Where a state rate is unavailable, the federal minimum still applies.
analysis_states = pd.DataFrame({
    "STATEFIP": sorted(df["STATEFIP"].unique())
})
analysis_years = pd.DataFrame({
    "YEAR": range(2001, 2026)
})

mw_grid = (
    analysis_states
    .merge(analysis_years, how="cross")
    .merge(
        state_minimum[["STATEFIP", "YEAR", "state_minimum"]],
        on=["STATEFIP", "YEAR"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        federal_minimum,
        on="YEAR",
        how="left",
        validate="many_to_one",
    )
)

if mw_grid["federal_minimum"].isna().any():
    missing_years = sorted(
        mw_grid.loc[
            mw_grid["federal_minimum"].isna(), "YEAR"
        ].unique()
    )
    raise ValueError(
        f"Federal minimum wage missing for years: {missing_years}"
    )

# Use whichever is higher: the state or federal minimum wage.
# np.fmax uses the federal rate when a state rate is missing.
mw_grid["binding_minimum"] = np.fmax(
    mw_grid["state_minimum"],
    mw_grid["federal_minimum"],
)
mw_grid["log_minimum"] = np.log(mw_grid["binding_minimum"])

if mw_grid["binding_minimum"].isna().any():
    missing_keys = mw_grid.loc[
        mw_grid["binding_minimum"].isna(),
        ["STATEFIP", "YEAR"],
    ]
    raise ValueError(
        "Binding minimum wage is missing for these state-years:\n"
        + missing_keys.to_string(index=False)
    )


# ---------------------------------------------------------------------
# Test 2: Is higher minimum-wage exposure associated with a narrower penalty?
# ---------------------------------------------------------------------

# Merge annual state minimum wages onto the CPS data.
df_mw = df.merge(
    mw_grid[["STATEFIP", "YEAR", "binding_minimum", "log_minimum"]],
    on=["STATEFIP", "YEAR"],
    how="left",
    validate="many_to_one",
)

if df_mw["log_minimum"].isna().any():
    missing_keys = (
        df_mw.loc[
            df_mw["log_minimum"].isna(),
            ["STATEFIP", "YEAR"],
        ]
        .drop_duplicates()
        .sort_values(["STATEFIP", "YEAR"])
    )
    raise ValueError(
        "Minimum wage is still missing after the state-year merge:\n"
        + missing_keys.to_string(index=False)
    )

df_mw["out_x_log_minimum"] = (
    df_mw["outsourced"] * df_mw["log_minimum"]
)

# Allow the outsourcing differential to vary by state and year.
# The state-by-year fixed effects absorb common state-year wage conditions.
df_mw["os_state"] = (
    df_mw["outsourced"].astype(str)
    + "_"
    + df_mw["STATEFIP"].astype(str)
)
df_mw["os_year"] = (
    df_mw["outsourced"].astype(str)
    + "_"
    + df_mw["YEAR"].astype(str)
)

minimum_wage_formula = (
    "log_wage ~ out_x_log_minimum + "
    + " + ".join(COVARIATES)
    + " | STATEFIP^YEAR + os_state + os_year"
)

minimum_wage_rows = []

for occupation in ["janitor", "guard"]:
    sample = df_mw[df_mw["occupation"] == occupation].copy()
    fit_mw = fit_model(minimum_wage_formula, sample)
    result = coefficient_row(fit_mw, "out_x_log_minimum")

    minimum_wage_rows.append({
        "occupation": occupation,
        "test": "Outsourced x log binding minimum wage",
        **result,
        "interpretation": (
            "A positive estimate is consistent with a narrower penalty "
            "at higher minimum wages."
        ),
    })

minimum_wage_results = pd.DataFrame(minimum_wage_rows)
minimum_wage_path = OUT / "hypothesis_minwage_tests.csv"
minimum_wage_results.to_csv(minimum_wage_path, index=False)

print("\nMinimum-wage interaction")
print(minimum_wage_results.round(4).to_string(index=False))
print(f"\nSaved: {minimum_wage_path}")

/Users/kq/opt/anaconda3/envs/outsourcing/lib/python3.12/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 3 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/kq/opt/anaconda3/envs/outsourcing/lib/python3.12/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 3 singleton fixed effect(s) dropped from the model.
  warnings.warn(



Trend tests
occupation                                               test  estimate     se      t  statistic  df      p     n
   janitor Period differences from 2001-2005 are jointly zero       NaN    NaN    NaN    16.0338 4.0 0.0030 60356
   janitor                Linear trend in penalty, per decade    0.0177 0.0059 2.9878        NaN NaN 0.0031 60356
     guard Period differences from 2001-2005 are jointly zero       NaN    NaN    NaN    25.0144 4.0 0.0000 24143
     guard                Linear trend in penalty, per decade    0.0219 0.0082 2.6675        NaN NaN 0.0081 24143

Saved: /Users/kq/Projects/outsourcing-penalty/outputs/py/hypothesis_trend_tests.csv

Minimum-wage interaction
occupation                                  test  estimate     se      t      p     n                                                                     interpretation
   janitor Outsourced x log binding minimum wage    0.0177 0.0357 0.4971 0.6195 60356 A positive estimate is consistent with a narrower p

/Users/kq/opt/anaconda3/envs/outsourcing/lib/python3.12/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 3 singleton fixed effect(s) dropped from the model.
  warnings.warn(
